# LeetCode 3410 — Maximize Subarray Sum After Removing All Occurrences of One Element bằng Python

- Link: https://leetcode.com/problems/maximize-subarray-sum-after-removing-all-occurrences-of-one-element/
- Độ khó: **Hard**
- Chủ đề: **Kadane**, **cây phân đoạn** lưu `(tổng, max đoạn con, max tiền tố, max hậu tố)`, **tổng tiền tố + chia khối theo các lần xuất hiện** + **sparse table** (RMQ)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSubarraySum(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu "xóa mọi lần xuất hiện của `x`" tương đương "đặt các phần tử `x` thành **0**" khi đáp án không âm — và xử lý riêng trường hợp mọi phần tử âm.
2. Dùng **cây phân đoạn** với nút `(sum, best, pre, suf)`: mỗi giá trị `x` cập nhật điểm các vị trí của nó về 0, đọc gốc, rồi khôi phục — tổng số cập nhật `O(n)`.
3. Cách nhanh hơn: với `x` cố định, `Q_x[i] = P[i] − x · C_x[i]`; max đoạn con = `max_{l<r} (Q_x[r] − Q_x[l])`. `C_x` hằng trên các **khối** giữa hai lần xuất hiện ⇒ chỉ cần max/min của `P` trên mỗi khối (sparse table).
4. Tổng số khối trên mọi `x` là `O(n)` ⇒ `O(n log n)` tiền xử lý + `O(n)` truy vấn.
5. Nhớ điều kiện "mảng còn lại khác rỗng".

## 2. Đề bài & ví dụ

Được làm **tối đa một lần**: chọn `x` (sao cho mảng còn khác rỗng) và xóa **mọi** phần tử bằng `x`. Trả về tổng đoạn con lớn nhất (khác rỗng) trên mọi mảng có thể thu được.

**Ví dụ 1:** `[-3,2,-2,-1,3,-2,3]` → `7` (xóa `-2`: `[-3,2,-1,3,3]`, đoạn `2,-1,3,3`).

**Ví dụ 2:** `[1,2,3,4]` → `10` (không xóa).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `|nums[i]| ≤ 10^6`.

In [1]:
from typing import List
from collections import defaultdict

examples = [([-3, 2, -2, -1, 3, -2, 3], 7), ([1, 2, 3, 4], 10)]
for a, exp in examples:
    print(a, '-> kỳ vọng', exp)

[-3, 2, -2, -1, 3, -2, 3] -> kỳ vọng 7
[1, 2, 3, 4] -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Kadane

In [2]:
def kadane(a):
    best = cur = a[0]
    for x in a[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best

print(kadane([-3, 2, -2, -1, 3, -2, 3]), kadane([-3, 2, -1, 3, 3]), kadane([-5, -2, -7]))

4 7 -2


### 3.2. Nút cây phân đoạn cho max đoạn con

Mỗi nút giữ `(sum, best, pre, suf)`; gộp trái `A` và phải `B`:

- `sum = A.sum + B.sum`; `pre = max(A.pre, A.sum + B.pre)`; `suf = max(B.suf, B.sum + A.suf)`;
- `best = max(A.best, B.best, A.suf + B.pre)`.

In [3]:
def leaf(v):
    return (v, v, v, v)

def merge(A, B):
    return (A[0] + B[0], max(A[1], B[1], A[3] + B[2]), max(A[2], A[0] + B[2]), max(B[3], B[0] + A[3]))

node = leaf(-3)
for v in [2, -2, -1, 3, -2, 3]:
    node = merge(node, leaf(v))
print('gộp tuần tự:', node, '-> best =', node[1])

gộp tuần tự: (0, 4, 0, 4) -> best = 4


### 3.3. Xóa ≡ đặt bằng 0 (khi đáp án ≥ 0)

Đặt các phần tử `x` thành `0` không đổi tổng của bất kỳ đoạn nào sau khi bỏ chúng. Khác biệt duy nhất: đoạn chỉ gồm các số 0 (ứng với đoạn **rỗng**) cho tổng 0. Nếu mảng có phần tử `≥ 0`, đáp án đã ≥ 0 nên đoạn "giả" này không làm sai. Nếu **mọi** phần tử âm: xóa không giúp được gì, đáp án là `max(nums)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chỉ có ích khi xóa một giá trị **âm** nằm "chen" giữa các phần dương. Ví dụ 1: xóa `-2` nối `2, -1, 3` và `3` lại: `2 − 1 + 3 + 3 = 7`. Xóa `-1` hoặc `-3` không nối được gì tốt hơn 4.

Với tổng tiền tố `P` (`P[0] = 0`), đoạn `[l, r)` sau khi xóa `x` có tổng `P[r] − P[l] − x · (số lần x trong đoạn)`.

In [4]:
a = examples[0][0]
for x in sorted(set(a)):
    rest = [v for v in a if v != x]
    print(f"xóa {x:2d}: {rest} -> {kadane(rest)}")

xóa -3: [2, -2, -1, 3, -2, 3] -> 4
xóa -2: [-3, 2, -1, 3, 3] -> 7
xóa -1: [-3, 2, -2, 3, -2, 3] -> 4
xóa  2: [-3, -2, -1, 3, -2, 3] -> 4
xóa  3: [-3, 2, -2, -1, -2] -> 2


## 5. Cách 1 — Brute Force: thử từng giá trị, chạy Kadane

In [5]:
def max_sum_brute(nums):
    ans = kadane(nums)
    for x in set(nums):
        rest = [v for v in nums if v != x]
        if rest:
            ans = max(ans, kadane(rest))
    return ans

for a, exp in examples:
    print(max_sum_brute(a), '| kỳ vọng', exp)

7 | kỳ vọng 7
10 | kỳ vọng 10


**Độ phức tạp:** `O(n · D)` (`D` số giá trị khác nhau) — tới `10^{10}`.

## 6. Ý tưởng thuật toán / bước đột phá

Cố định `x`, gọi `C[i]` = số lần `x` trong `nums[0..i)`. Với `Q[i] = P[i] − x · C[i]`, tổng đoạn `[l, r)` sau khi xóa `x` là `Q[r] − Q[l]`. Cần `max_{l < r} (Q[r] − Q[l])`.

Các chỉ số tiền tố được chia thành **khối** theo `C`: khối `c` gồm các `i` có `C[i] = c` (giữa lần xuất hiện thứ `c` và `c + 1`). Trong khối `c`, `Q = P − x·c`. Với `l` ở khối `c1` và `r` ở khối `c2 > c1`: giá trị tốt nhất `= (max P trên khối c2 − x·c2) − (min P trên khối c1 − x·c1)` — quét các khối, giữ min bên trái. Trường hợp cùng khối là đoạn **không chứa `x`** — đã có trong Kadane của mảng gốc.

Max/min `P` trên một khối bằng **sparse table** `O(1)`. Tổng số khối `Σ_x (cnt_x + 1) = n + D` ⇒ `O(n log n)` tổng.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cây phân đoạn: đặt các vị trí của `x` thành 0 (theo gợi ý)

In [6]:
class MaxSubTree:
    def __init__(self, a):
        self.size = 1
        while self.size < len(a):
            self.size *= 2
        NEG = float('-inf')
        self.empty = (0, NEG, NEG, NEG)
        self.t = [self.empty] * (2 * self.size)
        for i, v in enumerate(a):
            self.t[self.size + i] = leaf(v)
        for i in range(self.size - 1, 0, -1):
            self.t[i] = self.combine(self.t[2 * i], self.t[2 * i + 1])

    def combine(self, A, B):
        return (A[0] + B[0], max(A[1], B[1], A[3] + B[2]), max(A[2], A[0] + B[2]), max(B[3], B[0] + A[3]))

    def set(self, i, v):
        i += self.size
        self.t[i] = leaf(v)
        i //= 2
        while i:
            self.t[i] = self.combine(self.t[2 * i], self.t[2 * i + 1])
            i //= 2

    def best(self):
        return self.t[1][1]

def max_sum_segtree(nums):
    if max(nums) < 0:
        return max(nums)
    tree = MaxSubTree(nums)
    ans = tree.best()
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    for x, occ in pos.items():
        if x >= 0 or len(occ) == len(nums):
            continue                         # xóa số không âm không bao giờ có lợi
        for i in occ:
            tree.set(i, 0)
        ans = max(ans, tree.best())
        for i in occ:
            tree.set(i, x)
    return ans

for a, exp in examples:
    print(max_sum_segtree(a), '| kỳ vọng', exp)

7 | kỳ vọng 7
10 | kỳ vọng 10


### 7.2. Cách 3 — Khối theo lần xuất hiện + sparse table (lời giải chính)

In [7]:
def build_sparse(P):
    m = len(P)
    mn, mx = [P[:]], [P[:]]
    j = 1
    while (1 << j) <= m:
        a, b, h = mn[-1], mx[-1], 1 << (j - 1)
        mn.append([min(a[i], a[i + h]) for i in range(m - (1 << j) + 1)])
        mx.append([max(b[i], b[i + h]) for i in range(m - (1 << j) + 1)])
        j += 1
    def qmin(l, r):
        k = (r - l + 1).bit_length() - 1
        return min(mn[k][l], mn[k][r - (1 << k) + 1])
    def qmax(l, r):
        k = (r - l + 1).bit_length() - 1
        return max(mx[k][l], mx[k][r - (1 << k) + 1])
    return qmin, qmax

def max_sum_blocks(nums):
    n = len(nums)
    if max(nums) < 0:
        return max(nums)
    P = [0]
    for v in nums:
        P.append(P[-1] + v)
    qmin, qmax = build_sparse(P)
    ans = kadane(nums)
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    for x, occ in pos.items():
        if len(occ) == n:
            continue
        best_left = None
        start = 0
        for c, end in enumerate(occ + [n]):
            lo, hi = start, end                    # khối c: chỉ số tiền tố [lo, hi]
            if lo <= hi:
                M, m_ = qmax(lo, hi) - x * c, qmin(lo, hi) - x * c
                if best_left is not None:
                    ans = max(ans, M - best_left)
                best_left = m_ if best_left is None else min(best_left, m_)
            start = end + 1
    return ans

for a, exp in examples:
    print(max_sum_blocks(a), '| kỳ vọng', exp)

7 | kỳ vọng 7
10 | kỳ vọng 10


### 7.3. Cách 4 — Chỉ thử các giá trị **âm**, mỗi giá trị chạy Kadane

Cắt tỉa đơn giản của cách 1: xóa một giá trị không âm không bao giờ có lợi, nên chỉ thử các `x < 0`. Vẫn `O(n · D⁻)` tệ nhất (`D⁻` = số giá trị âm khác nhau), nhưng là bước trung gian dễ kiểm tra.

In [8]:
def max_sum_pruned_brute(nums):
    ans = kadane(nums)
    for x in set(nums):
        if x >= 0:
            continue
        rest = [v for v in nums if v != x]
        if rest:
            ans = max(ans, kadane(rest))
    return ans

for a, exp in examples:
    print(max_sum_pruned_brute(a), '| kỳ vọng', exp)

7 | kỳ vọng 7
10 | kỳ vọng 10


Vì sao bỏ `x ≥ 0` an toàn: xóa số không âm chỉ có thể làm **giảm** tổng mọi đoạn (hoặc giữ nguyên), và không "nối" thêm được gì có lợi hơn việc giữ chúng lại.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 với `x = -2`.

In [9]:
def trace(nums, x):
    n = len(nums)
    P = [0]
    for v in nums:
        P.append(P[-1] + v)
    occ = [i for i, v in enumerate(nums) if v == x]
    print('P =', P, '| vị trí của', x, ':', occ)
    best_left, start, ans = None, 0, None
    for c, end in enumerate(occ + [n]):
        lo, hi = start, end
        blockP = P[lo:hi + 1]
        M, m_ = max(blockP) - x * c, min(blockP) - x * c
        cand = M - best_left if best_left is not None else None
        if cand is not None:
            ans = cand if ans is None else max(ans, cand)
        print(f"khối c={c}: chỉ số tiền tố [{lo}, {hi}], P = {blockP}, max Q = {M}, min Q = {m_}, ứng viên {cand}")
        best_left = m_ if best_left is None else min(best_left, m_)
        start = end + 1
    print('tốt nhất khi xóa', x, ':', ans, '| Kadane trực tiếp:', kadane([v for v in nums if v != x]))

trace(examples[0][0], -2)

P = [0, -3, -1, -3, -4, -1, -3, 0] | vị trí của -2 : [2, 5]
khối c=0: chỉ số tiền tố [0, 2], P = [0, -3, -1], max Q = 0, min Q = -3, ứng viên None
khối c=1: chỉ số tiền tố [3, 5], P = [-3, -4, -1], max Q = 1, min Q = -2, ứng viên 4
khối c=2: chỉ số tiền tố [6, 7], P = [-3, 0], max Q = 4, min Q = 1, ứng viên 7
tốt nhất khi xóa -2 : 7 | Kadane trực tiếp: 7


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maxSubarraySum(self, nums: List[int]) -> int:
        n = len(nums)
        mx_val = max(nums)
        if mx_val < 0:
            return mx_val
        P = [0] * (n + 1)
        for i, v in enumerate(nums):
            P[i + 1] = P[i] + v
        m = n + 1
        mn_t, mx_t = [P], [P]
        j = 1
        while (1 << j) <= m:
            a, b, h = mn_t[-1], mx_t[-1], 1 << (j - 1)
            mn_t.append([min(a[i], a[i + h]) for i in range(m - (1 << j) + 1)])
            mx_t.append([max(b[i], b[i + h]) for i in range(m - (1 << j) + 1)])
            j += 1
        best = cur = nums[0]
        for v in nums[1:]:
            cur = v if cur < 0 else cur + v
            if cur > best:
                best = cur
        pos = {}
        for i, v in enumerate(nums):
            pos.setdefault(v, []).append(i)
        for x, occ in pos.items():
            if x >= 0 or len(occ) == n:
                continue
            left = None
            start = 0
            occ.append(n)
            for c, end in enumerate(occ):
                if start <= end:
                    k = (end - start + 1).bit_length() - 1
                    hi = max(mx_t[k][start], mx_t[k][end - (1 << k) + 1]) - x * c
                    lo = min(mn_t[k][start], mn_t[k][end - (1 << k) + 1]) - x * c
                    if left is not None and hi - left > best:
                        best = hi - left
                    if left is None or lo < left:
                        left = lo
                start = end + 1
        return best

for a, exp in examples:
    got = Solution().maxSubarraySum(a)
    print(got, 'OK' if got == exp else 'SAI')

7 OK
10 OK


## 10. Vì sao thuật toán đúng

1. **Không xóa:** Kadane cho đáp án của mảng gốc.
2. **Xóa `x`:** mỗi đoạn con khác rỗng của mảng mới ↔ đoạn `[l, r)` của mảng gốc (mở rộng tối đa qua các phần tử `x` ở hai đầu không thay đổi tổng), tổng `Q_x[r] − Q_x[l]`. Ngược lại mọi `l < r` cho một giá trị đạt được hoặc là `0` từ đoạn chỉ gồm `x` (ứng với rỗng) — vô hại khi `max(nums) ≥ 0` (mục 3.3).
3. **Khối:** trong khối, `Q = P − x·c`; với `l`, `r` ở hai khối khác nhau ràng buộc `l < r` tự thỏa ⇒ tối ưu tách thành max/min theo khối. Cùng khối ⇒ đoạn không chứa `x` ⇒ đã nằm trong Kadane gốc.
4. **Chỉ xét `x < 0`** (bản nộp) và `cnt_x < n` (mảng còn khác rỗng).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng giá trị + Kadane | `O(n · D)` | `O(n)` |
| 2. Cây phân đoạn, đặt 0 | `O(n log n)` (mỗi vị trí cập nhật 2 lần) | `O(n)` |
| 3. Khối + sparse table | `O(n log n)` tiền xử lý + `O(n + D)` | `O(n log n)` |
| 4. Thử giá trị âm + Kadane | `O(n · D⁻)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'segtree': max_sum_segtree, 'blocks': max_sum_blocks, 'pruned': max_sum_pruned_brute,
         'submit': lambda a: Solution().maxSubarraySum(a)}
edge = [([-5], -5), ([-2, -2, -2], -2), ([-1, -3, -1], -1), ([5, -100, 5], 10), ([3, -1, 3, -1, 3], 9), ([0], 0), ([2, 2, 2], 6)]
for a, exp in examples + edge:
    assert max_sum_brute(a) == exp, (a, max_sum_brute(a))
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
rng = random.Random(3410)
cnt = 0
for _ in range(4000):
    n = rng.randint(1, 12)
    a = [rng.randint(-5, 5) if rng.random() < 0.7 else rng.randint(-5, -1) for _ in range(n)]
    exp = max_sum_brute(a)
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'mảng ngẫu nhiên.')

Khớp vét cạn trên 4000 mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [13]:
import time
n = 10 ** 5
cases = {
    'giá trị rộng': [rng.randint(-10 ** 6, 10 ** 6) for _ in range(n)],
    'giá trị hẹp -5..5': [rng.randint(-5, 5) for _ in range(n)],
    'dương xen một số âm lặp': [(-10 ** 6 if i % 3 == 0 else 10 ** 6 - i % 7) for i in range(n)],
    'toàn âm': [rng.randint(-10 ** 6, -1) for _ in range(n)],
}
for name, a in cases.items():
    t = time.time(); r = Solution().maxSubarraySum(a); t1 = time.time() - t
    t = time.time(); r2 = max_sum_blocks(a); t2 = time.time() - t
    assert r == r2
    print(f"{name:26s}: {r:>14d} | nộp bài {t1:.2f}s | cách 3 {t2:.2f}s")

a = cases['giá trị hẹp -5..5'][:30000]
t = time.time(); r3 = max_sum_segtree(a); t3 = time.time() - t
assert r3 == Solution().maxSubarraySum(a)
print(f"cây phân đoạn (cách 2) ở n = 30000: {t3:.2f}s (mỗi giá trị âm cập nhật điểm 2 lần; tổng O(n log n))")

giá trị rộng              :      128700305 | nộp bài 0.27s | cách 3 0.32s


giá trị hẹp -5..5         :          45088 | nộp bài 0.17s | cách 3 0.20s


dương xen một số âm lặp   :    66665800007 | nộp bài 0.16s | cách 3 0.19s
toàn âm                   :            -10 | nộp bài 0.00s | cách 3 0.00s
cây phân đoạn (cách 2) ở n = 30000: 0.11s (mỗi giá trị âm cập nhật điểm 2 lần; tổng O(n log n))


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đặt về 0 mà quên trường hợp mọi phần tử âm

In [14]:
def wrong_all_negative(nums):
    best = kadane(nums)
    for x in set(nums):
        if len([v for v in nums if v != x]) == 0:
            continue
        best = max(best, kadane([0 if v == x else v for v in nums]))   # SAI khi mọi phần tử âm
    return best

print('sai :', wrong_all_negative([-3, -1, -2]), '| đúng:', Solution().maxSubarraySum([-3, -1, -2]))

sai : 0 | đúng: -1


**Vì sao sai:** đoạn chỉ gồm số 0 (vị trí đã xóa) có tổng 0 — ứng với đoạn **rỗng**, không hợp lệ. Khi mọi phần tử âm đáp án phải là `max(nums)`.

### Lỗi 2 — Cho phép xóa khi mảng chỉ có một giá trị (mảng trở thành rỗng)

In [15]:
def wrong_allow_empty(nums):
    best = kadane(nums)
    for x in set(nums):
        rest = [v for v in nums if v != x]
        best = max(best, kadane(rest) if rest else 0)     # SAI: mảng rỗng không được phép
    return best

print('sai :', wrong_allow_empty([-2, -2, -2]), '| đúng:', Solution().maxSubarraySum([-2, -2, -2]))

sai : 0 | đúng: -2


**Vì sao sai:** đề yêu cầu `nums` còn khác rỗng sau khi xóa.

### Lỗi 3 — Chỉ xóa **một** lần xuất hiện (phần tử âm nhỏ nhất trong đoạn) thay vì mọi lần

In [16]:
def wrong_remove_one_occurrence(nums):
    # "xóa tối đa một phần tử" (bài khác: LC 1186)
    n = len(nums)
    keep, drop, best = nums[0], float('-inf'), nums[0]
    for v in nums[1:]:
        drop = max(drop + v, keep)
        keep = max(keep + v, v)
        best = max(best, keep, drop)
    return best

rng = random.Random(3)
ce = next(a for a in ([rng.randint(-4, 4) for _ in range(rng.randint(2, 8))] for _ in range(20000))
          if wrong_remove_one_occurrence(a) != max_sum_brute(a))
print('phản ví dụ:', ce, '| sai:', wrong_remove_one_occurrence(ce), '| đúng:', max_sum_brute(ce))

phản ví dụ: [4, 3, 2, -2, -1, -2, 4, 2] | sai: 12 | đúng: 14


**Vì sao sai:** đây là bài LC 1186 (xóa tối đa **một phần tử**). Bài này xóa **mọi** lần xuất hiện của **một giá trị** — có thể nối nhiều đoạn (như ví dụ 1 xóa hai số `-2`), nhưng cũng buộc xóa cả những lần xuất hiện không muốn xóa.

### Lỗi 4 — Gộp khối nhưng cho phép `l`, `r` cùng khối với hệ số `x·c` khác nhau (lệch chỉ số khối)

In [17]:
def wrong_block_index(nums):
    n = len(nums)
    if max(nums) < 0:
        return max(nums)
    P = [0]
    for v in nums:
        P.append(P[-1] + v)
    ans = kadane(nums)
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    for x, occ in pos.items():
        if len(occ) == n:
            continue
        best_left, start = None, 0
        for c, end in enumerate(occ + [n]):
            lo, hi = start, end + 1 if end < n else end          # SAI: khối lấn sang chỉ số tiền tố sau x
            M, m_ = max(P[lo:hi + 1]) - x * c, min(P[lo:hi + 1]) - x * c
            if best_left is not None:
                ans = max(ans, M - best_left)
            best_left = m_ if best_left is None else min(best_left, m_)
            start = end + 1
    return ans

rng = random.Random(4)
ce = next(a for a in ([rng.randint(-4, 4) for _ in range(rng.randint(2, 8))] for _ in range(20000))
          if wrong_block_index(a) != max_sum_brute(a))
print('phản ví dụ:', ce, '| sai:', wrong_block_index(ce), '| đúng:', max_sum_brute(ce))

phản ví dụ: [0, -3, 2] | sai: 5 | đúng: 2


**Vì sao sai:** chỉ số tiền tố `i` thuộc khối `c` ⇔ `C[i] = c` ⇔ `occ[c−1] < i ≤ occ[c]`. Lấn thêm một chỉ số làm dùng sai hệ số `x·c` cho chỉ số đó.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Max đoạn con sau khi thay đổi một tập vị trí" ⇒ cây phân đoạn `(sum, best, pre, suf)` + cập nhật điểm.
- "Xóa phần tử" ≈ "đặt bằng 0" khi đáp án không âm — nhớ trường hợp toàn âm.
- Tổng `Σ_x cnt_x = n` ⇒ duyệt mọi giá trị, mỗi giá trị theo các lần xuất hiện của nó là tuyến tính tổng.
- Max hiệu tiền tố với dịch chuyển hằng theo khối ⇒ RMQ trên khối.

## 15. Biến thể của bài

1. **LC 53 — Maximum Subarray:** Kadane.
2. **LC 1186 — xóa tối đa một phần tử** (khác bài này — xem lỗi 3).
3. Xóa mọi lần xuất hiện của **tối đa hai** giá trị ⇒ khó hơn nhiều (cặp giá trị).
4. Max **tích** đoạn con sau khi xóa một giá trị ⇒ cây phân đoạn lưu cả max/min tích.

In [18]:
# Biến thể 2 (LC 1186) để so sánh: xóa tối đa một phần tử
def maximum_sum_one_deletion(arr):
    keep, drop, best = arr[0], float('-inf'), arr[0]
    for v in arr[1:]:
        drop = max(drop + v, keep)
        keep = max(keep + v, v)
        best = max(best, keep, drop)
    return best

assert maximum_sum_one_deletion([1, -2, 0, 3]) == 4
assert maximum_sum_one_deletion([1, -2, -2, 3]) == 3
assert maximum_sum_one_deletion([-1, -1, -1, -1]) == -1
print('LC 1186 ví dụ: 4, 3, -1 — so với bài này trên [1,-2,-2,3]:', Solution().maxSubarraySum([1, -2, -2, 3]))

LC 1186 ví dụ: 4, 3, -1 — so với bài này trên [1,-2,-2,3]: 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử từng giá trị + Kadane | `O(nD)` | Oracle |
| 2 | Cây phân đoạn đặt 0 | `O(n log n)` | Theo gợi ý, hằng số lớn |
| 3 | Khối + sparse table | `O(n log n)` | **Nộp bài** |
| 4 | Chỉ giá trị âm + Kadane | `O(n D⁻)` | Cắt tỉa đơn giản |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[4, -1, 4]`.

**Bài 2 (tính tay):** `[2, -5, 3, -5, 4, -1]`.

**Bài 3 (điền TODO):** viết `best_value_to_remove(nums)` — trả về `x` nên xóa (hoặc `None` nếu không xóa tốt nhất).

**Bài 4 (tự viết lại):** viết lại cách 2 bằng cây phân đoạn **đệ quy** có truy vấn đoạn.

**Bài 5 (mini challenge):** biến thể 4 — max **tích** đoạn con sau khi xóa mọi lần xuất hiện của một giá trị (mảng nhỏ, vét cạn có cấu trúc).

In [19]:
def best_value_to_remove_todo(nums):
    # TODO
    raise NotImplementedError

def segtree_recursive_todo(nums):
    # TODO
    raise NotImplementedError

def max_product_after_removal_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Xóa `-1`. (2) `[4, 4]`. (3) `8`.

**Bài 2.** (1) Xóa `-5`: `[2, 3, 4, -1]`. (2) Đoạn `2 + 3 + 4 = 9`. (3) So với không xóa (4).

**Bài 3.** (1) Ghi lại `x` khi cập nhật đáp án trong cách 3. (2) So sánh với Kadane gốc. (3) Hòa ⇒ không xóa.

**Bài 4.** (1) Hàm `build`, `update`, `query`. (2) Nút như mục 3.2. (3) So với cách 2.

**Bài 5.** (1) Thử từng giá trị. (2) Max tích đoạn con: giữ cả max và min kết thúc tại vị trí. (3) Kiểm tra với mảng nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().maxSubarraySum([4, -1, 4])
a2 = Solution().maxSubarraySum([2, -5, 3, -5, 4, -1])
assert a1 == 8 and a2 == max_sum_brute([2, -5, 3, -5, 4, -1]) == 9
print('Bài 1:', a1, '| Bài 2:', a2)

def best_value_to_remove(nums):
    base = kadane(nums)
    best, who = base, None
    for x in sorted(set(nums)):
        rest = [v for v in nums if v != x]
        if rest and kadane(rest) > best:
            best, who = kadane(rest), x
    return who, best

print('Bài 3:', best_value_to_remove(examples[0][0]), best_value_to_remove(examples[1][0]))

Bài 1: 8 | Bài 2: 9
Bài 3: (-2, 7) (None, 10)


In [21]:
# Bài 4, 5
def segtree_recursive(nums):
    n = len(nums)
    t = [None] * (4 * n)
    def build(node, lo, hi):
        if lo == hi:
            t[node] = leaf(nums[lo]); return
        mid = (lo + hi) // 2
        build(2 * node, lo, mid); build(2 * node + 1, mid + 1, hi)
        t[node] = merge(t[2 * node], t[2 * node + 1])
    def update(node, lo, hi, i, v):
        if lo == hi:
            t[node] = leaf(v); return
        mid = (lo + hi) // 2
        if i <= mid: update(2 * node, lo, mid, i, v)
        else: update(2 * node + 1, mid + 1, hi, i, v)
        t[node] = merge(t[2 * node], t[2 * node + 1])
    if max(nums) < 0:
        return max(nums)
    build(1, 0, n - 1)
    ans = t[1][1]
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    for x, occ in pos.items():
        if x >= 0 or len(occ) == n:
            continue
        for i in occ: update(1, 0, n - 1, i, 0)
        ans = max(ans, t[1][1])
        for i in occ: update(1, 0, n - 1, i, x)
    return ans

def max_product_sub(a):
    best = hi = lo = a[0]
    for v in a[1:]:
        cand = (v, hi * v, lo * v)
        hi, lo = max(cand), min(cand)
        best = max(best, hi)
    return best

def max_product_after_removal(nums):
    ans = max_product_sub(nums)
    for x in set(nums):
        rest = [v for v in nums if v != x]
        if rest:
            ans = max(ans, max_product_sub(rest))
    return ans

def max_product_brute(nums):
    def allsub(a):
        return max(eval('*'.join(map(str, a[i:j]))) for i in range(len(a)) for j in range(i + 1, len(a) + 1))
    ans = allsub(nums)
    for x in set(nums):
        rest = [v for v in nums if v != x]
        if rest:
            ans = max(ans, allsub(rest))
    return ans

rng = random.Random(19)
for _ in range(300):
    a = [rng.randint(-4, 4) for _ in range(rng.randint(1, 8))]
    assert segtree_recursive(a) == max_sum_brute(a)
    assert max_product_after_removal(a) == max_product_brute(a)
print('Bài 4, 5 khớp vét cạn. Ví dụ tích [2, -3, 0, -2, 4]:', max_product_after_removal([2, -3, 0, -2, 4]))

Bài 4, 5 khớp vét cạn. Ví dụ tích [2, -3, 0, -2, 4]: 48


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if max(nums) < 0: return max(nums)
P = tiền tố; sparse table max/min trên P
ans = kadane(nums)
for x < 0 với cnt_x < n:
    duyệt khối c (chỉ số tiền tố giữa lần xuất hiện thứ c và c+1):
        ans = max(ans, (maxP(khối) - x*c) - min_{khối trước}(minP - x*c'))
```

**Checklist:**

- [ ] Trường hợp mọi phần tử âm ⇒ `max(nums)`?
- [ ] Không xóa nếu làm mảng rỗng?
- [ ] Xóa **mọi** lần xuất hiện (không phải một phần tử)?
- [ ] Biên khối: `occ[c−1] < i ≤ occ[c]` (chỉ số tiền tố)?
- [ ] Vẫn tính Kadane của mảng gốc (không xóa)?